# 02 — Build the Player-Season Analytical Dataset

Builds `data/processed/player_season.csv` (plus `exclusion_summary.csv`): one
row per player per season, for the scope decided in `DATA_PLAN.md`:

- **Competitions**: 14 Aug-May domestic top-flight leagues (cups,
  international competitions, and calendar-year leagues excluded). Each
  league is identified by its unique `competition_id`; the source dataset's
  `name` column is *not* unique (Russia and Ukraine are both `premier-liga`),
  so a readable unique `competition` label is assigned explicitly.
- **Seasons**: 2020-2024 (the 2020/21 through 2024/25 seasons).
- **Market value**: latest `player_valuations` record on or before
  July 31 of `season + 1`, **and no older than 365 days before that cutoff**
  (`valuation_age_days`). Older matches are excluded and counted.
- **Multi-club players**: stats summed across clubs/competitions played
  that season; `club`/`competition` labeled by whichever had the most
  minutes (the player's primary club that season).
- **Per-90 stats**: computed for every row, but only meaningful when
  `analysis_minutes_eligible` (>= 450 minutes). Low-minute rows are kept in
  the dataset, not deleted.

No modeling here — construction and exclusion bookkeeping only.

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
OUT = Path("../data/processed/player_season.csv")
OUT_EXCLUSIONS = Path("../data/processed/exclusion_summary.csv")
pd.set_option("display.max_columns", 50)

SEASONS = range(2020, 2025)  # 2020/21 .. 2024/25
VALUATION_CUTOFF_MONTH_DAY = (7, 31)  # July 31 of season + 1
MAX_VALUATION_AGE_DAYS = 365  # see section 6b for the evidence behind this
MIN_MINUTES_PER90 = 450  # see section 6b for the evidence behind this

In [2]:
players = pd.read_csv(RAW / "players.csv.gz", low_memory=False)
appearances = pd.read_csv(RAW / "appearances.csv.gz", low_memory=False)
player_valuations = pd.read_csv(RAW / "player_valuations.csv.gz", low_memory=False)
games = pd.read_csv(RAW / "games.csv.gz", low_memory=False)
clubs = pd.read_csv(RAW / "clubs.csv.gz", low_memory=False)
competitions = pd.read_csv(RAW / "competitions.csv.gz", low_memory=False)

games["date"] = pd.to_datetime(games["date"])
player_valuations["date"] = pd.to_datetime(player_valuations["date"])
players["date_of_birth"] = pd.to_datetime(players["date_of_birth"])

## 1. Determine the league scope (Aug-May domestic leagues with appearance data)

In [3]:
domestic_leagues = competitions[competitions["type"] == "domestic_league"]
league_games = games.merge(domestic_leagues[["competition_id"]], on="competition_id")
season_start_month = (
    league_games.groupby(["competition_id", "season"])["date"].min().dt.month
    .groupby("competition_id").median()
)
euro_calendar_candidates = season_start_month[season_start_month >= 6].index

# Not every Aug-May league that shows up in `games` also has rows in
# `appearances` -- check which candidates actually have appearance data
# before finalizing the scope, rather than assuming.
candidate_app_ids = appearances.drop(columns=["competition_id"]).merge(
    games[["game_id", "competition_id"]], on="game_id", how="left"
)["competition_id"].unique()
euro_calendar_ids = [c for c in euro_calendar_candidates if c in candidate_app_ids]
no_appearance_data = domestic_leagues[
    domestic_leagues["competition_id"].isin(set(euro_calendar_candidates) - set(euro_calendar_ids))
]

print(f"{len(euro_calendar_candidates)} leagues classified as Aug-May by season-start-month")
print(f"{len(euro_calendar_ids)} of those actually have appearance-level data -- these are the analysis scope")
print(f"{len(no_appearance_data)} classified leagues have no appearance rows at all (games/competitions only):")
no_appearance_data[["competition_id", "name", "country_name"]]

23 leagues classified as Aug-May by season-start-month
14 of those actually have appearance-level data -- these are the analysis scope
9 classified leagues have no appearance rows at all (games/competitions only):


,competition_id,name,country_name
0,A1,bundesliga,Austria
4,AUS1,a-league-men,Australia
9,C1,super-league,Switzerland
35,KR1,supersport-hnl,Croatia
43,PL1,pko-bp-ekstraklasa,Poland
47,RO1,superliga,Romania
52,SA1,saudi-pro-league,Saudi Arabia
56,SER1,super-liga-srbije,Serbia
60,TS1,chance-liga,Czech Republic


## 2. Join appearances to games, filter to scope

In [4]:
app = appearances.drop(columns=["competition_id"]).merge(
    games[["game_id", "season", "competition_id"]], on="game_id", how="left"
)

n_before = len(app)
app_scope = app[
    app["competition_id"].isin(euro_calendar_ids) & app["season"].isin(SEASONS)
].copy()
n_after = len(app_scope)
print(f"appearance rows: {n_before:,} total -> {n_after:,} in scope ({n_after/n_before:.1%})")

appearance rows: 1,894,350 total -> 602,228 in scope (31.8%)


## 3. Aggregate to player-season (summed across any clubs/competitions in scope)

In [5]:
season_totals = app_scope.groupby(["player_id", "season"]).agg(
    appearances=("appearance_id", "count"),
    total_minutes=("minutes_played", "sum"),
    goals=("goals", "sum"),
    assists=("assists", "sum"),
).reset_index()

zero_minute_seasons = (season_totals["total_minutes"] == 0).sum()
print(f"player-seasons with zero total minutes: {zero_minute_seasons}")

season_totals["goals_per_90"] = season_totals["goals"] / (season_totals["total_minutes"] / 90)
season_totals["assists_per_90"] = season_totals["assists"] / (season_totals["total_minutes"] / 90)
season_totals["goal_contributions_per_90"] = season_totals["goals_per_90"] + season_totals["assists_per_90"]
season_totals.head()

player-seasons with zero total minutes: 0


,player_id,season,appearances,total_minutes,goals,assists,goals_per_90,assists_per_90,goal_contributions_per_90
0,3333,2020,26,1061,0,1,0.0,0.084826,0.084826
1,3333,2021,24,850,0,1,0.0,0.105882,0.105882
2,3333,2022,31,893,0,1,0.0,0.100784,0.100784
3,3333,2023,15,778,0,2,0.0,0.231362,0.231362
4,3333,2024,4,173,0,0,0.0,0.000000,0.000000


## 4. Primary club/competition per player-season (most minutes)

Handles the 6.4% of player-seasons with more than one club without
duplicating rows — see `DATA_PLAN.md` for the reasoning.

In [6]:
minutes_by_club = (
    app_scope.groupby(["player_id", "season", "player_club_id", "competition_id"])["minutes_played"]
    .sum().reset_index()
)
primary = (
    minutes_by_club.sort_values("minutes_played", ascending=False)
    .drop_duplicates(subset=["player_id", "season"], keep="first")
    .rename(columns={"player_club_id": "club_id", "competition_id": "primary_competition_id"})
    [["player_id", "season", "club_id", "primary_competition_id"]]
)

club_counts = app_scope.groupby(["player_id", "season"])["player_club_id"].nunique()
print(f"multi-club player-seasons: {(club_counts > 1).sum():,} of {len(club_counts):,} ({(club_counts > 1).mean():.1%})")

primary = primary.merge(clubs[["club_id", "name"]], on="club_id", how="left").rename(columns={"name": "club"})
primary.head()

multi-club player-seasons: 2,097 of 32,966 (6.4%)


,player_id,season,club_id,primary_competition_id,club
0,226012,2022,3709,ES1,Getafe CF
1,258919,2024,873,GB1,Crystal Palace
2,121483,2021,13,ES1,Atlético de Madrid
3,111873,2020,405,GB1,Aston Villa
4,144128,2021,940,ES1,Celta de Vigo


## 4b. Competition identity

The source `competitions.name` is not unique. Check the in-scope leagues, then
assign an explicit readable label per `competition_id`.

In [7]:
scope_comps = competitions[competitions["competition_id"].isin(euro_calendar_ids)][
    ["competition_id", "name", "country_name"]
].sort_values("name")
print("in-scope leagues sharing a source name:")
print(scope_comps[scope_comps.duplicated("name", keep=False)].to_string(index=False))

LEAGUE_LABELS = {
    "ES1": "LaLiga", "IT1": "Serie A", "GB1": "Premier League", "TR1": "Süper Lig",
    "PO1": "Liga Portugal", "L1": "Bundesliga", "NL1": "Eredivisie", "FR1": "Ligue 1",
    "UKR1": "Ukrainian Premier League", "RU1": "Russian Premier League",
    "BE1": "Belgian Pro League", "SC1": "Scottish Premiership",
    "GR1": "Greek Super League", "DK1": "Danish Superliga",
}
assert set(LEAGUE_LABELS) == set(euro_calendar_ids), "label map must cover exactly the in-scope leagues"
assert len(set(LEAGUE_LABELS.values())) == len(LEAGUE_LABELS), "labels must be unique"

comp_lookup = scope_comps.assign(competition=scope_comps["competition_id"].map(LEAGUE_LABELS))[
    ["competition_id", "competition", "country_name"]
].rename(columns={"country_name": "country"})
primary = primary.merge(comp_lookup, left_on="primary_competition_id", right_on="competition_id", how="left")
primary = primary.drop(columns=["primary_competition_id"])
comp_lookup.sort_values("competition")

in-scope leagues sharing a source name:
competition_id         name country_name
           RU1 premier-liga       Russia
          UKR1 premier-liga      Ukraine


,competition_id,competition,country
5,BE1,Belgian Pro League,Belgium
36,L1,Bundesliga,Germany
17,DK1,Danish Superliga,Denmark
39,NL1,Eredivisie,Netherlands
31,GR1,Greek Super League,Greece
23,ES1,LaLiga,Spain
44,PO1,Liga Portugal,Portugal
27,FR1,Ligue 1,France
29,GB1,Premier League,England
49,RU1,Russian Premier League,Russia


## 5. Join player attributes and derive age

In [8]:
player_info = players[["player_id", "name", "date_of_birth", "position", "sub_position"]].rename(
    columns={"name": "player_name"}
)

player_season = season_totals.merge(primary, on=["player_id", "season"], how="left")
player_season = player_season.merge(player_info, on="player_id", how="left")

season_start = pd.to_datetime(player_season["season"].astype(str) + "-08-01")
player_season["age"] = ((season_start - player_season["date_of_birth"]).dt.days / 365.25).round(1)
player_season.head()

,player_id,season,appearances,total_minutes,goals,assists,goals_per_90,assists_per_90,goal_contributions_per_90,club_id,club,competition_id,competition,country,player_name,date_of_birth,position,sub_position,age
0,3333,2020,26,1061,0,1,0.0,0.084826,0.084826,31,Liverpool FC,GB1,Premier League,England,James Milner,1986-01-04,Midfield,Central Midfield,34.6
1,3333,2021,24,850,0,1,0.0,0.105882,0.105882,31,Liverpool FC,GB1,Premier League,England,James Milner,1986-01-04,Midfield,Central Midfield,35.6
2,3333,2022,31,893,0,1,0.0,0.100784,0.100784,31,Liverpool FC,GB1,Premier League,England,James Milner,1986-01-04,Midfield,Central Midfield,36.6
3,3333,2023,15,778,0,2,0.0,0.231362,0.231362,1237,Brighton & Hove Albion,GB1,Premier League,England,James Milner,1986-01-04,Midfield,Central Midfield,37.6
4,3333,2024,4,173,0,0,0.0,0.000000,0.000000,1237,Brighton & Hove Albion,GB1,Premier League,England,James Milner,1986-01-04,Midfield,Central Midfield,38.6


## 6. Join season-level market value

Latest `player_valuations` record on or before July 31 of `season + 1`.

In [9]:
def latest_valuation_by_cutoff(valuations: pd.DataFrame, seasons) -> pd.DataFrame:
    rows = []
    for s in seasons:
        cutoff = pd.Timestamp(year=s + 1, month=VALUATION_CUTOFF_MONTH_DAY[0], day=VALUATION_CUTOFF_MONTH_DAY[1])
        sub = valuations[valuations["date"] <= cutoff].sort_values("date")
        latest = sub.drop_duplicates(subset="player_id", keep="last")[["player_id", "market_value_in_eur", "date"]]
        latest = latest.rename(columns={"date": "valuation_date"})
        latest["season"] = s
        rows.append(latest)
    return pd.concat(rows, ignore_index=True)

season_valuations = latest_valuation_by_cutoff(player_valuations, SEASONS)
player_season = player_season.merge(season_valuations, on=["player_id", "season"], how="left")
cutoff = pd.to_datetime((player_season["season"] + 1).astype(str) + "-07-31")
player_season["valuation_age_days"] = (cutoff - player_season["valuation_date"]).dt.days
player_season[["player_name", "season", "market_value_in_eur", "valuation_date", "valuation_age_days"]].head()

,player_name,season,market_value_in_eur,valuation_date,valuation_age_days
0,James Milner,2020,3000000.0,2021-06-08,53.0
1,James Milner,2021,2000000.0,2022-06-15,46.0
2,James Milner,2022,1500000.0,2023-06-20,41.0
3,James Milner,2023,1000000.0,2024-05-27,65.0
4,James Milner,2024,1000000.0,2025-05-30,62.0


## 6b. Inspect before locking the two quality rules

**Valuation staleness.** Look at how old the matched valuation is
(`valuation_age_days` = days between the valuation and the July-31 cutoff).

**Minutes threshold for per-90 stats.** Per-90 rates from a handful of
minutes are noise (one goal in one minute is "90 goals per 90"). Compare how
many rows survive at each candidate threshold and where the rate stops being
dominated by tiny samples.

In [10]:
age = player_season["valuation_age_days"].dropna()
print("valuation_age_days quantiles:")
print(age.quantile([.5, .9, .95, .99, 1.0]).round(0).to_string())
bins = [0, 30, 60, 90, 120, 180, 270, 365, 540, 730, 10_000]
print("\nrows by valuation age (days):")
print(pd.cut(age, bins).value_counts().sort_index().to_string())
stale = player_season["valuation_age_days"] > MAX_VALUATION_AGE_DAYS
print(f"\nage > {MAX_VALUATION_AGE_DAYS} days: {stale.sum()} rows; by season:")
print(player_season[stale]["season"].value_counts().sort_index().to_string())

valuation_age_days quantiles:
0.50      55.0
0.90      71.0
0.95      76.0
0.99     421.0
1.00    2472.0

rows by valuation age (days):
valuation_age_days
(0, 30]           210
(30, 60]        22544
(60, 90]         8140
(90, 120]         456
(120, 180]        397
(180, 270]        271
(270, 365]         39
(365, 540]        292
(540, 730]         45
(730, 10000]       79

age > 365 days: 416 rows; by season:
season
2020     16
2022     10
2023      6
2024    384


In [11]:
rows = []
for t in [90, 180, 270, 450, 600, 900]:
    e = player_season[player_season["total_minutes"] >= t]
    att = e[e["position"] == "Attack"]
    rows.append({
        "min_minutes": t, "rows_kept": len(e), "share_kept": round(len(e) / len(player_season), 3),
        "max_goals_per_90": round(e["goals_per_90"].max(), 2),
        "attack_goals_per_90_p99": round(att["goals_per_90"].quantile(.99), 2),
        "attack_goals_per_90_std": round(att["goals_per_90"].std(), 3),
    })
print(f"rows with < 90 min: {(player_season['total_minutes'] < 90).sum():,};  < 450 min: {(player_season['total_minutes'] < 450).sum():,}")
pd.DataFrame(rows)

rows with < 90 min: 3,751;  < 450 min: 9,378


,min_minutes,rows_kept,share_kept,max_goals_per_90,attack_goals_per_90_p99,attack_goals_per_90_std
0,90,29215,0.886,3.40,1.01,0.243
1,180,27424,0.832,2.56,0.97,0.225
2,270,26047,0.790,1.94,0.94,0.217
3,450,23588,0.716,1.50,0.93,0.208
4,600,21717,0.659,1.50,0.93,0.205
5,900,18437,0.559,1.50,0.91,0.200


**Decisions (documented in `DATA_PLAN.md`):**

- **`MAX_VALUATION_AGE_DAYS = 365`.** The bulk of matches are 30-120 days
  before the cutoff (the June update round); beyond ~270 days the count
  drops to a thin tail of rows whose value was not refreshed for the whole
  season. That tail is concentrated in 2024 (384 of 416 rows); the reason
  was not investigated. A 365-day cap is a simple "issued during or just after the
  season" rule. It is a practical cap, not a claim that 365 is uniquely
  correct.
- **`MIN_MINUTES_PER90 = 450`** (five full matches). Attacker goals-per-90
  spread and 99th percentile flatten out from roughly 450 minutes upward and
  the implausible maxima disappear, while ~72% of rows remain. Lower
  thresholds keep more rows but leave visibly noisy rates; higher ones
  discard many genuine rotation players. Low-minute rows are **kept** in the
  dataset (flagged, not dropped); only per-90 analyses filter on the flag.

## 7. Exclusions — inspect and apply, with counts logged

In [12]:
# Apply exclusions sequentially so each row is counted once, at the first rule it fails.
funnel = [("Initial player-seasons in scope (14 leagues, 2020-2024)", 0, len(player_season))]
df = player_season

def apply_rule(df, drop_mask, label):
    n_drop = int(drop_mask.sum())
    out = df[~drop_mask].copy()
    funnel.append((label, n_drop, len(out)))
    return out

df = apply_rule(df, df["position"].isna() | (df["position"] == "Missing"), "Missing position")
df = apply_rule(df, df["date_of_birth"].isna(), "Missing date of birth")
df = apply_rule(df, df["market_value_in_eur"].isna(), "No valuation on/before July 31 cutoff")
df = apply_rule(df, df["valuation_age_days"] > MAX_VALUATION_AGE_DAYS, f"Valuation older than {MAX_VALUATION_AGE_DAYS} days at cutoff")
df = apply_rule(df, df["total_minutes"] == 0, "Zero total minutes")
assert not df.duplicated(subset=["player_id", "season"]).any(), "duplicate (player_id, season) keys"

df["analysis_minutes_eligible"] = df["total_minutes"] >= MIN_MINUTES_PER90
n_eligible = int(df["analysis_minutes_eligible"].sum())

exclusion_summary = pd.DataFrame(funnel, columns=["step", "rows_dropped", "rows_remaining"])
exclusion_summary.loc[len(exclusion_summary)] = [
    f"Of final rows: eligible for per-90 analysis (>= {MIN_MINUTES_PER90} min; flagged, not dropped)",
    len(df) - n_eligible, n_eligible,
]
exclusion_summary.to_csv(OUT_EXCLUSIONS, index=False)
exclusion_summary

,step,rows_dropped,rows_remaining
0,"Initial player-seasons in scope (14 leagues, 2...",0,32966
1,Missing position,9,32957
2,Missing date of birth,5,32952
3,No valuation on/before July 31 cutoff,490,32462
4,Valuation older than 365 days at cutoff,416,32046
5,Zero total minutes,0,32046
6,Of final rows: eligible for per-90 analysis (>...,8718,23328


In [13]:
clean = df
print(f"rows dropped: {len(player_season) - len(clean):,}")
print(f"rows remaining: {len(clean):,}")
print(f"eligible for per-90 analysis: {n_eligible:,} ({n_eligible / len(clean):.1%})")

rows dropped: 920
rows remaining: 32,046
eligible for per-90 analysis: 23,328 (72.8%)


Rows with no valuation by the cutoff are the largest single drop (mostly
fringe squad players never valued). Stale valuations are the second-largest
(see 6b). Counts are sequential: each row is counted at the first rule it
fails. Saved to `data/processed/exclusion_summary.csv`.

## 8. Final column selection and save

In [14]:
final_cols = [
    "player_id", "player_name", "season", "age", "position", "sub_position",
    "club", "competition_id", "competition", "country",
    "appearances", "total_minutes", "goals", "assists",
    "goals_per_90", "assists_per_90", "goal_contributions_per_90", "analysis_minutes_eligible",
    "market_value_in_eur", "valuation_date", "valuation_age_days",
]
player_season_final = clean[final_cols].sort_values(["season", "player_id"]).reset_index(drop=True)

OUT.parent.mkdir(parents=True, exist_ok=True)
player_season_final.to_csv(OUT, index=False)
print(f"saved {len(player_season_final):,} rows to {OUT}")
player_season_final.head()

saved 32,046 rows to ../data/processed/player_season.csv


,player_id,player_name,season,age,position,sub_position,club,competition_id,competition,country,appearances,total_minutes,goals,assists,goals_per_90,assists_per_90,goal_contributions_per_90,analysis_minutes_eligible,market_value_in_eur,valuation_date,valuation_age_days
0,3333,James Milner,2020,34.6,Midfield,Central Midfield,Liverpool FC,GB1,Premier League,England,26,1061,0,1,0.000000,0.084826,0.084826,True,3000000.0,2021-06-08,53.0
1,3455,Zlatan Ibrahimović,2020,38.8,Attack,Centre-Forward,AC Milan,IT1,Serie A,Italy,19,1497,15,2,0.901804,0.120240,1.022044,True,4000000.0,2021-06-03,58.0
2,3875,Jermain Defoe,2020,37.8,Attack,Centre-Forward,Rangers FC,SC1,Scottish Premiership,Scotland,11,369,2,1,0.487805,0.243902,0.731707,False,800000.0,2021-06-28,33.0
3,4188,Ricardo Quaresma,2020,36.8,Attack,Right Winger,Vitória Guimarães SC,PO1,Liga Portugal,Portugal,28,1621,4,6,0.222085,0.333128,0.555213,True,300000.0,2021-05-21,71.0
4,4311,Maarten Stekelenburg,2020,37.9,Goalkeeper,Goalkeeper,Ajax Amsterdam,NL1,Eredivisie,Netherlands,12,1080,0,0,0.000000,0.000000,0.000000,True,550000.0,2021-05-27,65.0


## 9. Sanity checks on the final table

In [15]:
f = player_season_final
print("rows per season:")
print(f["season"].value_counts().sort_index().to_string())
print("\nposition counts:")
print(f["position"].value_counts().to_string())
print("\nleague counts:")
print(f["competition"].value_counts().to_string())
print("\nmarket_value_in_eur summary:")
print(f["market_value_in_eur"].describe().to_string())

assert not f.duplicated(subset=["player_id", "season"]).any()
assert (f[["goals", "assists", "total_minutes"]] >= 0).all().all()
assert (f["total_minutes"] > 0).all()
assert (f["market_value_in_eur"] > 0).all()
assert f["valuation_age_days"].between(0, MAX_VALUATION_AGE_DAYS).all()
assert f["competition"].nunique() == f["competition_id"].nunique() == 14
assert set(f["position"]) == {"Attack", "Midfield", "Defender", "Goalkeeper"}
assert f.drop(columns=["valuation_date"]).notna().all().all()
elig = f[f["analysis_minutes_eligible"]]
print(f"\nper-90 eligible: {len(elig):,}; max goals_per_90 among eligible = {elig['goals_per_90'].max():.2f}")
print("all sanity checks passed")

rows per season:
season
2020    6671
2021    5860
2022    6700
2023    6679
2024    6136

position counts:
position
Defender      10942
Midfield       9428
Attack         9116
Goalkeeper     2560

league counts:
competition
Süper Lig                   2775
Serie A                     2715
LaLiga                      2653
Premier League              2549
Liga Portugal               2546
Ligue 1                     2510
Eredivisie                  2327
Bundesliga                  2323
Belgian Pro League          2306
Russian Premier League      2165
Ukrainian Premier League    2074
Greek Super League          1972
Scottish Premiership        1606
Danish Superliga            1525

market_value_in_eur summary:
count    3.204600e+04
mean     5.447824e+06
std      1.168538e+07
min      2.000000e+04
25%      4.250000e+05
50%      1.300000e+06
75%      5.000000e+06
max      2.000000e+08

per-90 eligible: 23,328; max goals_per_90 among eligible = 1.50
all sanity checks passed
